# 06 — Protocole complet deux datasets

Wine Quality UCI reste l'expérience principale. Pima Indians Diabetes est utilisé comme dataset secondaire ; si le téléchargement échoue, le notebook utilise Breast Cancer Wisconsin comme fallback offline.

In [ ]:
# AML — Stacking et Blending — Protocole deux datasets
import sys, subprocess, pkgutil, warnings, random, time, os
warnings.filterwarnings("ignore")

def install_if_missing(package, import_name=None):
    import_name = import_name or package.replace("-", "_")
    if pkgutil.find_loader(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

for package, import_name in [
    ("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"), ("scikit-learn", "sklearn"),
    ("mlxtend", "mlxtend"), ("xgboost", "xgboost"), ("lightgbm", "lightgbm")
]:
    install_if_missing(package, import_name)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVC
from mlxtend.classifier import StackingCVClassifier

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBMClassifier = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
OUT = Path("outputs")
PLOTS = OUT / "plots"
PLOTS.mkdir(parents=True, exist_ok=True)

In [ ]:
UCI_RED_WINE_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
PIMA_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

def load_wine_quality_binary():
    df = pd.read_csv(UCI_RED_WINE_URL, sep=";")
    X = df.drop(columns=["quality"])
    y = (df["quality"] >= 6).astype(int)
    return X, y, "Wine Quality UCI"

def load_pima_or_fallback():
    cols = ["pregnancies","glucose","blood_pressure","skin_thickness","insulin","bmi","diabetes_pedigree","age","outcome"]
    try:
        df = pd.read_csv(PIMA_URL, names=cols)
        X = df.drop(columns=["outcome"])
        y = df["outcome"].astype(int)
        return X, y, "Pima Indians Diabetes"
    except Exception:
        data = load_breast_cancer(as_frame=True)
        return data.data, data.target.astype(int), "Breast Cancer Wisconsin fallback"

datasets = [load_wine_quality_binary(), load_pima_or_fallback()]
for X, y, name in datasets:
    print(name, X.shape, pd.Series(y).value_counts().to_dict())

In [ ]:
def build_preprocessor(X):
    num_cols = X.select_dtypes(include=["number"]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()
    num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)])

def wrap(model, X):
    return Pipeline([("prep", build_preprocessor(X)), ("model", model)])

def base_learners(X, seed=42):
    learners = [
        ("lr", wrap(LogisticRegression(max_iter=5000, random_state=seed), X)),
        ("svm", wrap(SVC(kernel="rbf", probability=True, random_state=seed), X)),
        ("knn", wrap(KNeighborsClassifier(n_neighbors=9), X)),
        ("rf", wrap(RandomForestClassifier(n_estimators=250, random_state=seed, n_jobs=-1), X)),
        ("gb", wrap(GradientBoostingClassifier(random_state=seed), X)),
    ]
    if XGBClassifier is not None:
        learners.append(("xgb", wrap(XGBClassifier(n_estimators=250, learning_rate=0.05, max_depth=4, eval_metric="logloss", random_state=seed, n_jobs=-1), X)))
    if LGBMClassifier is not None:
        learners.append(("lgbm", wrap(LGBMClassifier(n_estimators=250, learning_rate=0.05, random_state=seed, verbose=-1), X)))
    return learners

class CustomStackingClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, base_estimators, meta_estimator=None, n_splits=5, random_state=42):
        self.base_estimators = base_estimators
        self.meta_estimator = meta_estimator
        self.n_splits = n_splits
        self.random_state = random_state

    def fit(self, X, y):
        X = pd.DataFrame(X).reset_index(drop=True)
        y = pd.Series(y).reset_index(drop=True)
        self.names_ = [n for n, _ in self.base_estimators]
        self.classes_ = np.unique(y)
        Z = np.zeros((len(X), len(self.base_estimators)))
        skf = StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        for tr, va in skf.split(X, y):
            for j, (name, est) in enumerate(self.base_estimators):
                m = clone(est)
                m.fit(X.iloc[tr], y.iloc[tr])
                Z[va, j] = m.predict_proba(X.iloc[va])[:, 1]
        self.oof_meta_features_ = pd.DataFrame(Z, columns=self.names_)
        self.meta_estimator_ = clone(self.meta_estimator or LogisticRegression(max_iter=5000, random_state=self.random_state))
        self.meta_estimator_.fit(self.oof_meta_features_, y)
        self.fitted_base_estimators_ = []
        for name, est in self.base_estimators:
            m = clone(est)
            m.fit(X, y)
            self.fitted_base_estimators_.append((name, m))
        return self

    def transform(self, X):
        X = pd.DataFrame(X).reset_index(drop=True)
        Z = np.zeros((len(X), len(self.fitted_base_estimators_)))
        for j, (name, m) in enumerate(self.fitted_base_estimators_):
            Z[:, j] = m.predict_proba(X)[:, 1]
        return pd.DataFrame(Z, columns=self.names_)

    def predict(self, X):
        return self.meta_estimator_.predict(self.transform(X))

    def predict_proba(self, X):
        return self.meta_estimator_.predict_proba(self.transform(X))

class CustomBlendingClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, base_estimators, meta_estimator=None, blend_size=0.25, random_state=42):
        self.base_estimators = base_estimators
        self.meta_estimator = meta_estimator
        self.blend_size = blend_size
        self.random_state = random_state

    def fit(self, X, y):
        X = pd.DataFrame(X).reset_index(drop=True)
        y = pd.Series(y).reset_index(drop=True)
        X_base, X_blend, y_base, y_blend = train_test_split(X, y, test_size=self.blend_size, stratify=y, random_state=self.random_state)
        self.names_ = [n for n, _ in self.base_estimators]
        self.fitted_base_estimators_ = []
        Z = np.zeros((len(X_blend), len(self.base_estimators)))
        for j, (name, est) in enumerate(self.base_estimators):
            m = clone(est)
            m.fit(X_base, y_base)
            Z[:, j] = m.predict_proba(X_blend)[:, 1]
            self.fitted_base_estimators_.append((name, m))
        self.meta_estimator_ = clone(self.meta_estimator or LogisticRegression(max_iter=5000, random_state=self.random_state))
        self.meta_estimator_.fit(Z, y_blend)
        return self

    def _Z(self, X):
        X = pd.DataFrame(X).reset_index(drop=True)
        Z = np.zeros((len(X), len(self.fitted_base_estimators_)))
        for j, (name, m) in enumerate(self.fitted_base_estimators_):
            Z[:, j] = m.predict_proba(X)[:, 1]
        return Z

    def predict(self, X):
        return self.meta_estimator_.predict(self._Z(X))

    def predict_proba(self, X):
        return self.meta_estimator_.predict_proba(self._Z(X))

In [ ]:
def evaluate(model, Xtr, ytr, Xte, yte, model_name, dataset_name):
    t0 = time.perf_counter()
    model.fit(Xtr, ytr)
    train_time = time.perf_counter() - t0
    pred = model.predict(Xte)
    row = {
        "dataset": dataset_name,
        "model": model_name,
        "accuracy": accuracy_score(yte, pred),
        "f1_macro": f1_score(yte, pred, average="macro"),
        "precision_macro": precision_score(yte, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(yte, pred, average="macro", zero_division=0),
        "train_time_sec": train_time,
    }
    if hasattr(model, "predict_proba"):
        try:
            row["roc_auc"] = roc_auc_score(yte, model.predict_proba(Xte)[:, 1])
        except Exception:
            row["roc_auc"] = np.nan
    return row

def build_models(X, seed=42):
    base = base_learners(X, seed)
    models = {name: model for name, model in base}
    models["hard_voting"] = VotingClassifier(estimators=base, voting="hard", n_jobs=-1)
    models["soft_voting"] = VotingClassifier(estimators=base, voting="soft", n_jobs=-1)
    models["custom_stacking"] = CustomStackingClassifier(base, n_splits=5, random_state=seed)
    models["sklearn_stacking"] = StackingClassifier(estimators=base, final_estimator=LogisticRegression(max_iter=5000, random_state=seed), cv=5, stack_method="predict_proba", n_jobs=-1)
    models["mlxtend_stackingcv"] = StackingCVClassifier(classifiers=[m for _, m in base], meta_classifier=LogisticRegression(max_iter=5000, random_state=seed), use_probas=True, cv=5, random_state=seed, n_jobs=-1)
    models["custom_blending"] = CustomBlendingClassifier(base, blend_size=0.25, random_state=seed)
    return models

all_results = []
for X, y, dataset_name in datasets:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
    for name, model in build_models(X, SEED).items():
        print("Training", dataset_name, name)
        all_results.append(evaluate(model, Xtr, ytr, Xte, yte, name, dataset_name))

results = pd.DataFrame(all_results).sort_values(["dataset","f1_macro"], ascending=[True, False])
display(results)
results.to_csv(OUT / "results_table.csv", index=False)

plt.figure(figsize=(13, 5))
sns.barplot(data=results, x="model", y="f1_macro", hue="dataset")
plt.xticks(rotation=35, ha="right")
plt.title("F1 macro — comparaison deux datasets")
plt.tight_layout()
plt.savefig(PLOTS / "barplot_f1_macro_two_datasets.png", dpi=160)
plt.show()

In [ ]:
# Étude 30 seeds sur Wine Quality uniquement pour respecter l'exigence principale.
X, y, dataset_name = datasets[0]
rows = []
for seed in range(30):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=seed)
    base = base_learners(X, seed)
    models = {
        "best_lr": base[0][1],
        "soft_voting": VotingClassifier(estimators=base, voting="soft", n_jobs=-1),
        "custom_stacking": CustomStackingClassifier(base, n_splits=5, random_state=seed),
        "sklearn_stacking": StackingClassifier(estimators=base, final_estimator=LogisticRegression(max_iter=5000), cv=5, stack_method="predict_proba", n_jobs=-1),
        "custom_blending": CustomBlendingClassifier(base, blend_size=0.25, random_state=seed),
    }
    for name, model in models.items():
        row = evaluate(model, Xtr, ytr, Xte, yte, name, dataset_name)
        row["seed"] = seed
        rows.append(row)

stability = pd.DataFrame(rows)
stability.to_csv(OUT / "stability_30_seeds.csv", index=False)

summary = stability.groupby("model").agg(
    mean_f1=("f1_macro","mean"),
    std_f1=("f1_macro","std"),
    min_f1=("f1_macro","min"),
    max_f1=("f1_macro","max"),
    mean_accuracy=("accuracy","mean"),
    mean_train_time_sec=("train_time_sec","mean"),
).sort_values("mean_f1", ascending=False)

display(summary)
summary.to_csv(OUT / "stability_summary.csv")

plt.figure(figsize=(12, 5))
order = summary.index.tolist()
sns.boxplot(data=stability, x="model", y="f1_macro", order=order)
plt.xticks(rotation=35, ha="right")
plt.title("Wine Quality — F1 macro sur 30 seeds")
plt.tight_layout()
plt.savefig(PLOTS / "boxplot_30_seeds.png", dpi=160)
plt.show()

## Discussion à remplir après exécution

1. Le stacking bat-il Best Single ?
2. Le stacking bat-il Voting ?
3. Le blending est-il plus instable ?
4. Les conclusions sont-elles cohérentes sur le second dataset ?
5. Quelle méthode choisir pour un déploiement ?